# 03 · Organoid development (5–8 months)
Dryad dataset (Sharf et al. 2022, Nat. Commun.): organoid L1 (array 2953), MaxOne, recorded at 5, 6, 7 and 8 months.

In [ ]:
import os # environment settings
from pathlib import Path # file paths
os.environ["HDF5_PLUGIN_PATH"] = str(Path.home() / "hdf5_plugin_path_maxwell") # MaxWell decompression plugin; before importing spikeinterface

import numpy as np # arrays and maths
import spikeinterface.extractors as se # file readers

data_dir = Path("/Volumes/ADATA HD710/github/hdmea-organoid-pipeline") # folder with the 4 recordings
ages = [5, 6, 7, 8] # organoid age in months; drop one to skip that file
files = {age: data_dir / f"Development_2953_{age}month.raw.h5" for age in ages} # age -> file path

for age, f in files.items(): # one block per recording
    print(f"--- {age} months: {f.name} ---") # header
    print("Exists:", f.exists()) # should be True
    rec = se.read_maxwell(f) # open (metadata only, fast)
    print(f"Channels: {rec.get_num_channels()}") # electrodes recorded
    print(f"Sampling rate: {rec.get_sampling_frequency():.0f} Hz") # expect 20000
    print(f"Duration: {rec.get_total_duration():.1f} s") # recording length
    print(f"Gain: {rec.get_channel_gains()[0]:.3f} µV per ADC count\n") # compare with the test file (6.294)

--- 5 months: Development_2953_5month.raw.h5 ---
Exists: True
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
Channels: 1020
Sampling rate: 20000 Hz
Duration: 179.8 s
Gain: 6.294 µV per ADC count

--- 6 months: Development_2953_6month.raw.h5 ---
Exists: True
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
Channels: 1020
Sampling rate: 20000 Hz
Duration: 180.2 s
Gain: 6.294 µV per ADC count

--- 7 months: Development_2953_7month.raw.h5 ---
Exists: True
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
Channels: 1020
Sampling rate: 20000 Hz
Duration: 179.8 s
Gain: 6.294 µV per ADC count

--- 8 months: Development_2953_8month.raw.h5 ---
Exists: True
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwe

## 1. Sort one recording

In [ ]:
import spikeinterface as si # core SpikeInterface
import spikeinterface.preprocessing as spre # cleaning tools
import spikeinterface.sorters as ss # spike sorters

age_sort = 5 # which recording to sort: 5, 6, 7 or 8
sort_root = data_dir / "sorting" # where sorting results are saved
sort_dir = sort_root / f"sc2_{age_sort}month" # one folder per age
si.set_global_job_kwargs(n_jobs=4, progress_bar=True) # 4 cores + progress bars

rec = se.read_maxwell(files[age_sort]) # open the chosen recording
rec_s = spre.unsigned_to_signed(rec) # uint16 -> signed, ready for the sorter
rec_s = rec_s.select_channels(rec_s.get_channel_ids()) # keep only the 1020 declared channels -> drops the 8 extra columns
print(rec_s) # check: 1020 channels, ~180 s

x = rec_s.get_traces(start_frame=0, end_frame=10) # read just 10 samples
print("Data shape:", x.shape, "| channels declared:", rec_s.get_num_channels()) # columns should equal 1020

run_sorting = False # True = sort now (long!); set to False afterwards to just load the saved result
if run_sorting: # only when you ask for it
    sorting = ss.run_sorter(
        "spykingcircus2", # which sorter
        rec_s, # signed, unfiltered: SC2 filters itself
        folder=sort_dir, # where sorting results are saved
        remove_existing_folder=True, # overwrite old results
        verbose=True, # print progress
        deterministic_peaks_detection=True, # no early stopping
        apply_motion_correction=False, # organoid slice on the chip barely moves
    )
else:
    sorting = ss.read_sorter_folder(sort_dir) # load the saved result
print(f"Units found: {len(sorting.get_unit_ids())}") # number of neurons

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
ChannelSliceRecording: 1020 channels - 20.0kHz - 1 segments - 3,596,600 samples 
                       179.83s (3.00 minutes) - int16 dtype - 6.83 GiB
Data shape: (10, 1020) | channels declared: 1020
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
Units found: 134


## 2. Sort the remaining ages

In [ ]:
import time # to measure run time

run_loop = False # True = sort the ages below (long!); False = skip, results are already saved
ages_todo = [6, 7, 8] # recordings to sort when run_loop is True
for age in (ages_todo if run_loop else []): # loops over nothing when run_loop is False
    t0 = time.time() # start the clock
    rec_a = se.read_maxwell(files[age]) # open this age's recording
    rec_a = spre.unsigned_to_signed(rec_a) # uint16 -> signed
    rec_a = rec_a.select_channels(rec_a.get_channel_ids()) # keep only the 1020 real channels (drops the 8 extra rows)
    print(f"\n=== {age} months: {rec_a.get_num_channels()} channels, {rec_a.get_total_duration():.1f} s ===") # check before sorting
    sorting_a = ss.run_sorter(
        "spykingcircus2", # which sorter
        rec_a, # signed, unfiltered
        folder=sort_root / f"sc2_{age}month", # same folder naming as the 5-month one
        remove_existing_folder=True, # overwrite old results
        verbose=False, # less output, since we run 3 in a row
        deterministic_peaks_detection=True, # no early stopping
        apply_motion_correction=False, # slice barely moves
    )
    print(f"{age} months: {len(sorting_a.get_unit_ids())} units, {(time.time() - t0) / 60:.1f} min") # result + run time